# Reachability-based BLAs for intersection crossings

An executable derivation in the style of [protocol.ipynb](protocol.ipynb). The vehicle model,
the intersection, the reachability construction and the protocol stages are all defined
below. A requested crossing is followed through its numerical interpretation, client
contract, quotient, admission, offers, bargaining and exact commitment. Cells tagged
`definitions` are the method: `ltms_bla.scenario.run_scenario` compiles exactly those cells
for the serial runner. Modules hold only reusable machinery: the pyspect-based PDE solver
(`ltms_reachability`), contract records, raster geometry, caching, artifact storage and plots.

**Kernel.** Python 3.13 with `pyspect[hj_reachability]==1.2`, because JAX has no Python 3.14
wheel on Intel macOS (see `docs/LTMS_BLA_EXAMPLE.md`).

**Evidence labels.** Every claim below carries one of four strengths.

| Label | Reasoning | Example |
|:--|:--|:--|
| **[algebra]** | contract identity by Boolean normalization | quotient; $K_\kappa\Rightarrow S_a$ |
| **[semantic]** | implication from reference/body containment and the fixed client policy | $H_\kappa\Rightarrow a_s$; $S_a\Rightarrow B_j\land S_u$ |
| **[numerical]** | TLT and corridor membership on the declared grid | entry value $\le 0$ |
| **[independent]** | nominal execution checks and analytic bounds | body gap; speed bound |

**Scope.** A finite nominal example: Bicycle5D, exact initial states, straight zero-input
references, a finite departure library, time-shifted and rotated base corridors, serial
requests A–E. The numerical TLT is neither an owner promise nor a controller certificate.
No general feedback synthesis, uncertain tracking, complete spatial satisfiability or
per-request HJ replanning is claimed.

In [1]:
from dataclasses import replace
from importlib.metadata import version
from itertools import combinations
from pathlib import Path
from IPython.display import Markdown, display
import os
import sys
import warnings
import numpy as np

warnings.filterwarnings('ignore', message='IProgress not found')  # tqdm without ipywidgets
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'src' / 'ltms_reachability').is_dir())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from scipy.ndimage import maximum_filter
from pyspect import ContLTL, Set, TLT, UNTIL
from pyspect.systems.hj_reachability import Bicycle5D
import ltms_reachability
from ltms_reachability import Axis, GridSpec, LTMSPlanner, TimeWindow, WindowPolicy, create_4way
from ltms_reachability.levelsets import rectangle, hyperplane, union, intersection
from ltms_reachability.windows import window_mask
from ltms_bla.checks import boxes_covered, minimum_body_gap
from ltms_bla.presentation import (MEANINGS, table, candidate_rows, results_table,
                                   plot_scenario, export_report)
from ltms_bla.records import implies, refines, ledger_symbols
from ltms_bla.scenario import (cached, array_digest, source_digest, run_scenario,
                               build_report, outcome_summary)
from ltms_bla.static_figure import export_figure
from protocol.presentation import display_timeline

OUTPUT = Path(os.environ.get('LTMS_OUTPUT', ROOT / 'artifacts' / 'ltms_blas'))
OUTPUT.mkdir(parents=True, exist_ok=True)

In [2]:
from dataclasses import asdict
from fractions import Fraction as F
import json
import numpy as np
import interval_logic as il
from protocol import Contract, CommitResult, affine_policies, normalize
from ltms_bla.bargaining import load_bargain
from ltms_bla.checks import aligned_disjoint
from ltms_bla.records import (CrossingService, CrossingRequest, Candidate, ServiceAdmission, Admission,
                              CorridorOffer, standing, snapshot_id, digest, certify_service,
                              certify_compatibility, commit_exact)
from ltms_bla.scenario import verify_artifact, write_artifact

# The protocol.ipynb function body; only its offer-eligibility validator differs.
bargain = load_bargain(ROOT)

## 1. Model

**Actors.** The manager (owner) controls allocation sets $r_j(t)\subseteq\mathbb R^2$. Vehicle
$j$ (client) controls its inputs and follows its declared policy from a specified entry state.

**Vehicle.** Bicycle5D with state $z=(x,y,h,\delta,v)$ and input $u=(u_1,u_2)$:
$$\dot x=v\cos h,\quad \dot y=v\sin h,\quad \dot h=v\tan\delta/L,\quad \dot\delta=u_1,\quad \dot v=u_2 .$$
The chart is route-local and nonperiodic; westbound motion is its rotation by $\pi$. This is
not the thesis's global grid. The body is a toy rectangle inside a declared disk; the sizes
are illustrative, not a measured vehicle.

In [3]:
WHEELBASE = .32                                                                # L [m]
CONTROLS = dict(min_steer=-.1, max_steer=.1, min_accel=-.15, max_accel=.15)    # u1, u2 bounds
CHART = dict(h=(-.5, .5), d=(-.15, .15), v=(.3, .6))                           # route-local chart
VMAX = CHART['v'][1]


def bicycle5d(z, u):
    x, y, h, d, v = z
    return np.array([v*np.cos(h), v*np.sin(h), v*np.tan(d)/WHEELBASE, u[0], u[1]])


BODY_SIZE = (.32, .16)  # length, width [m]
BODY_RADIUS = .18       # declared disk: Body(z) is this disk centred at (x, y)
TRACKING = .01          # extra allocation allowance; not a tracking proof
assert BODY_RADIUS >= np.hypot(BODY_SIZE[0]/2, BODY_SIZE[1]/2)

**Workspace and task.** Coordinates are eastbound and route-local. The whole body must stay
in the eastbound lane, and the task is complete when the reference point reaches the exit
and the body has left the managed square.

In [4]:
LANE = (-.8, 0.)  # eastbound lane, y-range for the whole body
MANAGED = 1.2     # managed square [-1.2, 1.2]^2
GOAL_X = 1.4      # the exit begins here


def body_in_lane(y, radius=BODY_RADIUS):
    return LANE[0] <= y - radius and y + radius <= LANE[1]


def body_clear(x, radius=BODY_RADIUS):
    return x >= GOAL_X and x - radius > MANAGED

**The fixed client.** For service $\theta$ with route, departure library $D_\theta$ and
deadline $d_\theta$, the reference for $s\in D_\theta$ starts at one exact entry state and
applies $u=(0,0)$. It is *present* on $[s,s+7]$; its lease covers $[s,s+8]$. With
$\xi_s$ the reference trajectory,
$$a_s(r_j):=\forall t\in[s,s+7]:\ \operatorname{Body}(\xi_s(t))\subseteq r_j(t),\qquad
S_a(\theta):=\bigvee_{s\in D_\theta:\ s+7\le d_\theta}a_s .$$
The client departs at the earliest eligible $s$ whose $a_s$ its accepted guarantee implies.
The rule reads only the agreement, never unobserved future allocation behaviour. Library,
entry state and policy are frozen for every offer of one selected service.

In [5]:
SPEED = .45
ENTRY = (-1.6, -.4, 0., 0., SPEED)  # route-local (x, y, h, δ, v) at departure
PRESENCE = 7                        # present on [s, s+7]
LEASE = 8                           # lease on [s, s+8]
CLEAR_WINDOW = (PRESENCE, LEASE)    # the TLT's local target window


def reference(tau, speed=SPEED, entry=ENTRY):
    # Solution of bicycle5d with h = δ = 0 and u = 0.
    x, y, h, d, _ = entry
    return (x + speed*tau, y, h, d, speed)


def eligible(service):
    # S_a ranges over departures whose reference clears by the deadline.
    return tuple(s for s in service.departures if s + PRESENCE <= service.deadline)


def tlt_evidence_applies(departure, deadline):
    # The local window [7, 8] lies in [s, d]; then the TLT strengthens "clear by d".
    return departure + CLEAR_WINDOW[1] <= deadline


def speed_bound_excludes(departure, deadline):
    # Independent necessary condition: the reference point must travel from entry to GOAL_X.
    return F(str(VMAX))*(deadline - departure) < F(str(GOAL_X)) - F(str(ENTRY[0]))


def select_departure(service, promised):
    # Fixed client rule: the earliest eligible departure whose a_s the agreement promises.
    return min((s for s in eligible(service) if s in promised), default=None)

| Predicate | Meaning | Arguments | Controlled by | Horizon | Evaluation / evidence |
|:--|:--|:--|:--|:--|:--|
| $H_\kappa$ | $r_j(t)\supseteq R_\kappa(t)$: the allocation covers the entire inflated reservation | allocation | owner | lease $[s,s+8]$ | owner promise; $R_\kappa$ is constructed in §2 |
| $a_s$ | $\operatorname{Body}(\xi_s(t))\subseteq r_j(t)$ | allocation; declared $\xi_s$ | owner | presence $[s,s+7]$ | consequence of $H_\kappa$: swept reference-body boxes lie in $R_\kappa$ (§2) |
| $S_a(\theta)$ | $\bigvee a_s$ over eligible $s$ | allocation | owner | $[\min D_\theta, d_\theta]$ | definition; Boolean algebra |
| $B_j$ | $\operatorname{Body}(z_j(t))\subseteq r_j(t)$ while present | allocation; vehicle trajectory | client, given $r_j$ | presence $[s,s+7]$ | client policy: following a covered $\xi_s$ gives containment |
| $S_u(\theta)$ | body in the lane, then clear of the square with $x\ge1.4$, by $d_\theta$ | vehicle trajectory | client | $[s,d_\theta]$ | straight-motion arithmetic below; the TLT in §2 is a separate feasibility check |

**Why $S_a$ suffices [semantic].** Suppose $a_s$ holds and the vehicle follows $\xi_s$ exactly
(perfect initial state and dynamics). On $[s,s+7]$ its body is
$\operatorname{Body}(\xi_s(t))\subseteq r_j(t)$, which is $B_j$. Straight motion keeps the body
in the lane and clears the square by $s+7\le d_\theta$, which is $S_u(\theta)$. Hence
$C_{\rm client}=\langle S_a\mid B_j\land S_u\rangle$ holds for this client, whichever offer is
later accepted. The next cell checks the arithmetic, the reference against the dynamics, and
`bicycle5d` against the solver's own Bicycle5D; references that fail are shown as well.

In [6]:
def client_argument(departure, deadline, speed=SPEED, lateral=ENTRY[1]):
    x, y, *_ = reference(PRESENCE, speed, (ENTRY[0], lateral, 0., 0., speed))
    return dict(body_in_lane=body_in_lane(y), clears=body_clear(x), timely=departure + PRESENCE <= deadline,
                exit_x=x)

references = {'declared: 0.45 m/s, s=0, d=9': client_argument(0, 9),
              'slower: 0.35 m/s, s=0, d=9': client_argument(0, 9, speed=.35),
              'late: s=3, d=9': client_argument(3, 9),
              'off-centre: y=-0.7, s=0, d=9': client_argument(0, 9, lateral=-.7)}
table(['Reference [semantic]', 'Body in lane', 'x at s+7 [m]', 'Body clears square', 's+7 ≤ d'],
      [(name, a['body_in_lane'], a['exit_x'], a['clears'], a['timely']) for name, a in references.items()])
assert [a['body_in_lane'] and a['clears'] and a['timely'] for a in references.values()] == [True, False, False, False]

# The reference solves bicycle5d, sampled every 0.025 s [independent].
taus = np.linspace(0, PRESENCE, 281)
path = np.array([reference(t) for t in taus])
residual = np.abs(np.diff(path, axis=0)
                  - np.diff(taus)[:, None]*np.array([bicycle5d(z, (0, 0)) for z in path[:-1]])).max()
# bicycle5d is the model the HJ solver integrates.
solver = Bicycle5D(**CONTROLS, wheelbase=WHEELBASE)
samples = np.random.default_rng(0).uniform([-2, -2, -.5, -.15, .3, -.1, -.15], [2, 2, .5, .15, .6, .1, .15], (50, 7))
same_model = all(np.allclose(bicycle5d(z[:5], z[5:]), np.asarray(solver.open_loop_dynamics(z[:5], 0.))
                             + np.asarray(solver.control_jacobian(z[:5], 0.)) @ z[5:], atol=1e-6) for z in samples)
assert residual < 1e-12 and same_model
display(Markdown(f'Reference residual {residual:.1e}; `bicycle5d` equals pyspect Bicycle5D at 50 sampled (z, u): {same_model}.'))

| Reference [semantic] | Body in lane | x at s+7 [m] | Body clears square | s+7 ≤ d |
| --- | --- | --- | --- | --- |
| declared: 0.45 m/s, s=0, d=9 | True | 1.55 | True | True |
| slower: 0.35 m/s, s=0, d=9 | True | 0.85 | False | True |
| late: s=3, d=9 | True | 1.55 | True | False |
| off-centre: y=-0.7, s=0, d=9 | False | 1.55 | True | True |

Reference residual 2.7e-16; `bicycle5d` equals pyspect Bicycle5D at 50 sampled (z, u): True.

## 2. Environment, task and offered corridor

The numerical grid covers $(t,x,y,h,\delta,v)$ over a 9 s horizon in 0.25 s steps. `planner`
wraps pyspect's `TVHJImpl` with the Bicycle5D model above; it does no solving until asked.

In [7]:
DT = .25
grid_spec = GridSpec((Axis('t', 0, 9, 37), Axis('x', -2, 2, 31), Axis('y', -2, 2, 61),
                      Axis('h', *CHART['h'], 9), Axis('d', *CHART['d'], 5), Axis('v', *CHART['v'], 7)),
                     accuracy='medium')
planner = LTMSPlanner.bicycle5d(grid_spec, wheelbase=WHEELBASE, **CONTROLS)
grid, times = planner.grid, planner.times
x, y, h, d, v = (np.asarray(c) for c in grid.coordinate_vectors)


def backend(spec):
    # Everything a cached solve depends on besides its input arrays.
    return dict(grid=spec.fingerprint(dynamics=dict(CONTROLS, wheelbase=WHEELBASE)), format=4,
                solver=source_digest(ltms_reachability),
                versions={n: version(n) for n in ('pyspect', 'hj-reachability', 'jax', 'numpy')})

**The intersection.** The eastbound pieces of the copied four-way LTMS map, as level sets over
$(x,y,h,\delta,v)$ (inside $\Leftrightarrow$ value $\le 0$). They constrain the *reference
point*, not the body. The center is the octagon $|x|,|y|\le1.2$, $|x|+|y|\le2$; `road_e` is the
eastbound lane with heading and speed limits, or the center.

In [8]:
AXIS = dict(x=0, y=1, h=2, d=3, v=4)


def box(**bounds):
    return rectangle(grid, [lo for lo, _ in bounds.values()], [hi for _, hi in bounds.values()],
                     axes=[AXIS[name] for name in bounds])


speed_limit = box(v=(.3, .6))
center = intersection(hyperplane(grid, [-1, -1], [-1, -1]), hyperplane(grid, [1, -1], [1, -1]),
                      hyperplane(grid, [1, 1], [1, 1]), hyperplane(grid, [-1, 1], [-1, 1]),
                      box(x=(-MANAGED, MANAGED), y=(-MANAGED, MANAGED)), speed_limit)
road_e = union(intersection(box(y=LANE, h=(-np.pi/5, np.pi/5)), speed_limit), center)
entry_w = intersection(box(x=(-2, -1.4), y=(-.68, -.12)), road_e)
exit_e = intersection(box(x=(GOAL_X, 2), y=(-.68, -.12)), road_e)
LANE_STRIP = (-.46, -.34)  # the offered corridor keeps the reference point in this strip
lane_strip = box(y=LANE_STRIP)

# Fidelity: the same inside sets as the legacy map code.
legacy = create_4way(grid, 'road_e', 'entry_w', 'exit_e')
assert all(np.array_equal(region <= 0, legacy[name] <= 0)
           for name, region in (('road_e', road_e), ('entry_w', entry_w), ('exit_e', exit_e)))

**Masks versus the full body [semantic, on the grid].** Which full-body facts do the
reference-point sets imply? Where they do not, the body checks below stay necessary.

In [9]:
def inside_values(region, axis):
    # Grid coordinates along one axis at which the region has an inside point.
    present = (region <= 0).any(axis=tuple(i for i in range(5) if i != AXIS[axis]))
    return dict(x=x, y=y)[axis][present]

relations = [
    ('lane strip ± body ⊆ lane', all(map(body_in_lane, LANE_STRIP)),
     'every corridor state keeps the whole body in the lane'),
    ('road_e ⊆ body-in-lane', all(map(body_in_lane, inside_values(road_e, 'y'))),
     'the road set alone does not keep the body in the lane'),
    ('exit_e ⇒ body clear of the square', all(map(body_clear, inside_values(exit_e, 'x'))),
     'reaching the goal set clears the whole body in x'),
    ('exit_e ⊆ body-in-lane', all(map(body_in_lane, inside_values(exit_e, 'y'))),
     'the goal set alone does not keep the body in the lane')]
table(['Relation', 'Holds', 'Consequence'], relations)
assert [held for _, held, _ in relations] == [True, False, True, False]

| Relation | Holds | Consequence |
| --- | --- | --- |
| lane strip ± body ⊆ lane | True | every corridor state keeps the whole body in the lane |
| road_e ⊆ body-in-lane | False | the road set alone does not keep the body in the lane |
| exit_e ⇒ body clear of the square | True | reaching the goal set clears the whole body in x |
| exit_e ⊆ body-in-lane | False | the goal set alone does not keep the body in the lane |

**Task → TLT → HJ [numerical].** The supported physical task, in route-local time, is a
constrained until with a timed target:
$$\varphi_{\rm loc}:=\mathit{road}_e\ \mathcal U_{[7,8]}\ \mathit{exit}_e\quad\text{evaluated at }\tau=0 .$$
The goal is masked to $\tau\in[7,8]$ (value $+1$, i.e. empty, elsewhere); the tree is realized
by `TVHJImpl`, whose native until also intersects the target with the constraint at each
witness sample.

**Deadline relation [semantic].** In absolute time the TLT asks for the goal within
$[s+7,s+8]$. If $s+8\le d_\theta$, this *implies* "reach the goal by $d_\theta$": a sufficient
strengthening of the requested task, not an equivalence. If $s+7\le d_\theta<s+8$, the
reference is eligible for $S_a$ but the TLT evidence does not apply; such a template is
`UNRESOLVED`, never impossible.

The first run solves for a few minutes; `cached` stores each result under a key built from its
exact input arrays and `backend(grid_spec)`, and verifies the stored file's digest before reuse.

In [10]:
def timed_until(planner, constraint, target, window):
    mask = window_mask(planner.times, window).reshape(-1, 1, 1, 1, 1, 1)
    goal = np.where(mask, target, 1.).astype(planner.dtype)
    admissible = np.broadcast_to(constraint, planner.shape).astype(planner.dtype)
    tree = TLT(UNTIL('admissible', 'goal'), primitives=ContLTL,
               where=dict(admissible=Set(admissible), goal=Set(goal)))
    return tree, dict(admissible=array_digest(admissible), goal=array_digest(goal))


tree, inputs = timed_until(planner, road_e, exit_e, TimeWindow(*CLEAR_WINDOW))
tlt, tlt_record = cached(OUTPUT, 'tlt', dict(backend(grid_spec), **inputs),
                         lambda: dict(root=np.asarray(tree.realize(planner.impl))))
root = tlt['root']

entry = tuple(int(np.abs(c - e).argmin()) for c, e in zip((x, y, h, d, v), ENTRY))
TOLERANCE = 2e-5
table(['TLT item', 'Value'], [
    ('Tree', f'`{tree!r}`'),
    ('Target time samples τ [s]', ', '.join(f'{t:g}' for t in times[window_mask(times, TimeWindow(*CLEAR_WINDOW))])),
    ('Full entry state on the grid', [float(c[i]) for c, i in zip((x, y, h, d, v), entry)]),
    ('TLT value at the entry state', f'{root[(0, *entry)]:.5f} (inside if ≤ {TOLERANCE})'),
    ('Solve', f"{tlt_record['seconds']:.0f} s; cache used this run: {tlt_record['cache_hit']}")])
assert root[(0, *entry)] <= TOLERANCE

| TLT item | Value |
| --- | --- |
| Tree | `TLT(UNDER, ('F-UNTIL', '_0000000011bbfd55', '_0000000011bc2375'))` |
| Target time samples τ [s] | 7, 7.25, 7.5, 7.75, 8 |
| Full entry state on the grid | (-1.6, -0.4, 0, 0, 0.45) |
| TLT value at the entry state | -0.11535 (inside if ≤ 2e-05) |
| Solve | 45 s; cache used this run: True |

**Offer construction: the four LTMS passes [numerical].** `planner.analyze` computes pass 1, the
backward reach of `exit_e` under `road_e`; pass 2, pass 1 minus reserved space–time (no
reservations are given, so it *is* pass 1, the unreserved base); pass 3, the forward reach
from `entry_w` ∩ pass 2 in a sampled entry window; pass 4, the backward reach from pass 3 ∩
`exit_e` in a sampled exit window. The offered tube is pass 4 intersected with the lane strip.

In [11]:
def four_passes():
    result = planner.analyze(entry=entry_w, goal=exit_e, constraints=road_e,
                             entry_window=TimeWindow(0, .5), exit_window=TimeWindow(*CLEAR_WINDOW),
                             policy=WindowPolicy(0, .5, 0, 1))
    p = result.passes
    return dict(pass1=p['pass1'], pass3=p['pass3'], pass4=p['pass4'],
                pass2_is_pass1=np.array(p['pass2'] is p['pass1']),
                entry_window=np.array([result.entry_window.start, result.entry_window.end]),
                exit_window=np.array([result.exit_window.start, result.exit_window.end]))


passes, passes_record = cached(OUTPUT, 'passes', dict(backend(grid_spec), entry=array_digest(entry_w),
                               goal=array_digest(exit_e), constraints=array_digest(road_e),
                               windows=[[0, .5], list(CLEAR_WINDOW)], policy=[0, .5, 0, 1]), four_passes)
assert passes['pass2_is_pass1']
passes['pass2'] = passes['pass1']
corridor = np.maximum(passes['pass4'], lane_strip).astype(np.float32)

steps = {'pass1': 'backward reach of exit_e under road_e', 'pass2': 'pass 1 itself (no reservations)',
         'pass3': f"forward from entry_w ∩ pass 2, entry window {passes['entry_window'].tolist()} s",
         'pass4': f"backward from pass 3 ∩ exit_e, exit window {passes['exit_window'].tolist()} s",
         'corridor': f'pass 4 ∩ lane strip y ∈ {LANE_STRIP}'}
tubes = dict(passes, corridor=corridor)
table(['Tube', 'Restriction', 'Inside fraction', 'Value at entry'],
      [(name, step, f'{(tubes[name] <= 0).mean():.4f}', f'{tubes[name][(0, *entry)]:.4f}') for name, step in steps.items()])
display(Markdown(f"Passes: {passes_record['seconds']:.0f} s; cache used this run: {passes_record['cache_hit']}. "
                 f"**[numerical observation]** Exactly, the TLT set lies inside pass 1; on the grid "
                 f"{100*np.mean((root <= 0) & (passes['pass1'] > 0)):.3f} % of cells are inside the TLT tube only, "
                 "so separately computed tubes are never combined."))

| Tube | Restriction | Inside fraction | Value at entry |
| --- | --- | --- | --- |
| pass1 | backward reach of exit_e under road_e | 0.1330 | -0.1171 |
| pass2 | pass 1 itself (no reservations) | 0.1330 | -0.1171 |
| pass3 | forward from entry_w ∩ pass 2, entry window [0.0, 0.5] s | 0.0301 | -0.1171 |
| pass4 | backward from pass 3 ∩ exit_e, exit window [7.0, 8.0] s | 0.0182 | -0.0787 |
| corridor | pass 4 ∩ lane strip y ∈ (-0.46, -0.34) | 0.0021 | -0.0600 |

Passes: 125 s; cache used this run: True. **[numerical observation]** Exactly, the TLT set lies inside pass 1; on the grid 0.033 % of cells are inside the TLT tube only, so separately computed tubes are never combined.

**The concrete execution [independent, numerical].** The reference rollout from §1 must stay
inside the offered tube at every sample, keep its body in the lane and clear on time.

In [12]:
def nearest(axis, values):
    return np.abs(axis[:, None] - values[None, :]).argmin(axis=0)

along = corridor[nearest(times, taus), *(nearest(c, path[:, i]) for i, c in enumerate((x, y, h, d, v)))]
SERVICE_EVIDENCE = dict(
    entry_inside=bool(corridor[(0, *entry)] <= TOLERANCE and root[(0, *entry)] <= TOLERANCE),
    rollout_inside=bool(along.max() <= TOLERANCE),
    road_ok=all(map(body_in_lane, path[:, 1])),
    clear_ok=bool(body_clear(path[-1, 0]) and residual < 1e-8),
    input_ok=CONTROLS['min_steer'] <= 0 <= CONTROLS['max_steer'] and CONTROLS['min_accel'] <= 0 <= CONTROLS['max_accel'])
table(['Check', 'Value'], [('Corridor value at entry', f'{corridor[(0, *entry)]:.5f}'),
                           ('Max corridor value along ξ (0.025 s samples)', f'{along.max():.5f}'),
                           *SERVICE_EVIDENCE.items()])
assert all(SERVICE_EVIDENCE.values())

| Check | Value |
| --- | --- |
| Corridor value at entry | -0.06000 |
| Max corridor value along ξ (0.025 s samples) | -0.06000 |
| entry_inside | True |
| rollout_inside | True |
| road_ok | True |
| clear_ok | True |
| input_ok | True |

**The reservation $R_\kappa$ [numerical].** Project the tube onto the workspace at every time
sample (some $(h,\delta,v)$ inside), keep time, and inflate by the body radius, the allowance, a
half grid cell and one time slab's travel. Slab $i$ also reserves the projection at
$t_{i+1}$. Then every swept reference-body box must lie in reserved cells: that is the
evidence for $H_\kappa\Rightarrow a_s$. A reference whose presence leaves the reservation's
time range is uncovered.

In [13]:
occupied = (corridor <= 0).any(axis=(3, 4, 5))  # (t, x, y)
dx, dy = x[1] - x[0], y[1] - y[0]
margin = (BODY_RADIUS + TRACKING + dx/2 + VMAX*DT,
          BODY_RADIUS + TRACKING + dy/2 + VMAX*np.sin(CHART['h'][1])*DT)
kx, ky = int(np.ceil(margin[0]/dx)), int(np.ceil(margin[1]/dy))
padded = np.pad(occupied, ((0, 0), (kx+2, kx+2), (ky+2, ky+2)))  # room for bodies at the domain edge
slabs = padded.copy()
slabs[:-1] |= padded[1:]
reserved = maximum_filter(slabs.astype(np.uint8), size=(1, 2*kx+1, 2*ky+1), mode='constant') > 0
rx = x[0] + np.arange(-(kx+2), len(x)+kx+2)*dx
ry = y[0] + np.arange(-(ky+2), len(y)+ky+2)*dy


def covers_reference(delay=0.):
    # Swept body boxes of a reference departing `delay` after the reservation's time zero.
    if delay < times[0] - 1e-7 or delay + PRESENCE > times[-1] + 1e-7:
        return False
    boxes = []
    for i in range(len(times) - 1):
        start, stop = times[i] - delay, times[i+1] - delay
        if stop < -1e-7 or start > PRESENCE + 1e-7:
            continue
        a, b = (reference(np.clip(t, 0, PRESENCE))[0] for t in (start, stop))
        boxes.append((i, a - BODY_RADIUS, b + BODY_RADIUS, ENTRY[1] - BODY_RADIUS, ENTRY[1] + BODY_RADIUS))
    return boxes_covered(reserved, rx, ry, boxes)


SERVICE_EVIDENCE['reference_covered'] = covers_reference()
assert SERVICE_EVIDENCE['reference_covered']
base_arrays, base_record = cached(OUTPUT, 'base', dict(format=4, corridor=array_digest(corridor),
                                  reserved=array_digest(reserved)),
                                  lambda: dict(tube=corridor, reserved=reserved, rx=rx, ry=ry, times=times))
BASE = dict(reserved=reserved, dt=DT, path=base_record['path'], digest=base_record['digest'],
            service_checks=SERVICE_EVIDENCE, output=str(OUTPUT))
table(['Reservation', 'Value'], [('Margins (x, y) [m]', margin), ('Reserved cells', int(reserved.sum())),
                                 ('Swept reference body ⊆ reservation', SERVICE_EVIDENCE['reference_covered']),
                                 ('Hashed base artifact', Path(base_record['path']).name)])

| Reservation | Value |
| --- | --- |
| Margins (x, y) [m] | (0.4067, 0.2952) |
| Reserved cells | 7084 |
| Swept reference body ⊆ reservation | True |
| Hashed base artifact | base-b1e073184354007496ed.npz |

**Three roles, kept apart.**
- The *TLT* **[numerical]** says that from the full entry state some admissible input keeps
  the reference point in `road_e` until it reaches `exit_e` within $\tau\in[7,8]$ on this grid.
  It is an independent task-feasibility check: it does not check any offer's allocation, its
  witness is not the reference controller, and separate existential results are not assumed to
  share a trajectory.
- The *four passes and the lane strip* **[numerical]** construct the offered tube; its inflated
  projection is the reservation $R_\kappa$.
- The *explicit reference* **[independent]** is one concrete execution inside the tube whose
  swept body lies in $R_\kappa$ and which clears on time. It connects an offer to task execution.

An offer-constrained TLT solve would be a later extension; the finite-reference argument does
not need it.

## 3. Contracts, admission and offers

**Client and task [algebra].**
$$C_{\rm client}=\langle S_a\mid B_j\land S_u\rangle,\qquad R_{\rm task}=\langle\top\mid B_j\land S_u\rangle,$$
$$Q=R_{\rm task}/C_{\rm client}=\langle\neg S_a\lor(B_j\land S_u)\mid S_a\rangle,\qquad
C_{\rm cmt}^{+}=C_{\rm cmt}\wedge Q=\langle\top\mid L\land S_a\rangle .$$
`Contract` stores saturated guarantees; assertions check the computed identities.

**Admission.** Try the original service, then permitted adaptations in order; an adaptation
rebuilds client, task and quotient. Each template $\kappa$ is one departure: the base corridor
shifted to $s$ and rotated to the route. Qualification is a *sufficient* procedure within this
finite family. A template qualifies when its service evidence gives $H_\kappa\Rightarrow
a_s\Rightarrow S_a$ and its compatibility evidence gives an allocation satisfying
$L\land H_\kappa$; together they witness the owner obligation $L\land S_a$. This is neither
the charging booking solver nor a decision procedure for arbitrary spatial formulas, and
filtering templates never redefines $S_a$.

| Template status | Meaning |
|:--|:--|
| `QUALIFIED` | service and compatibility evidence both hold |
| `GEOMETRIC_CONFLICT` | template rejected: its reservation meets a standing one |
| `UNRESOLVED` | the numerical evidence does not apply ($s+8>d_\theta$) |
| `PROVED_IMPOSSIBLE` | the independent speed bound fails |

In [14]:
def service_contracts(service):
    # III-A: the fixed client and the joint task for one selected service.
    S_a = il.Or(service.access_for(s) for s in eligible(service))
    B, S_u = service.rule, service.task_atom
    return S_a, B, S_u, Contract(S_a, B & S_u), Contract(True, B & S_u)


def route_reservation(base, route):
    # Westbound is the eastbound map rotated by π: reverse both workspace axes.
    return base['reserved'] if route == 'east' else base['reserved'][:, ::-1, ::-1]


def qualify(service, departure, snapshot, ledger, base):
    # One finite template, checked against every standing reservation of the frozen snapshot.
    offer_id = digest(dict(service=service.key, departure=departure, snapshot=snapshot,
                           base=base['digest']))[:20]
    if speed_bound_excludes(departure, service.deadline):
        return Candidate(departure, offer_id, dict(
            status='PROVED_IMPOSSIBLE', evidence='independent analytic bound',
            reason=f'{GOAL_X - ENTRY[0]:g} m to the exit exceeds vmax · {service.deadline - departure} s'))
    if not tlt_evidence_applies(departure, service.deadline):
        return Candidate(departure, offer_id, dict(
            status='UNRESOLVED', evidence='numerical evidence does not apply',
            reason='local clearance window [s+7, s+8] is not within the deadline'))
    mine = route_reservation(base, service.route)
    comparisons = [dict(offer=b.offer.id, disjoint=aligned_disjoint(
                       mine, departure, route_reservation(base, b.offer.service.route),
                       b.offer.departure, base['dt']))
                   for b in ledger]
    if not all(c['disjoint'] for c in comparisons):
        return Candidate(departure, offer_id, dict(
            status='GEOMETRIC_CONFLICT', evidence='template rejected', reason='reservation meets a standing one',
            service=base['service_checks'], comparisons=comparisons))
    info = dict(id=offer_id, service=asdict(service), departure=departure, route=service.route,
                snapshot=snapshot, end=departure + LEASE, base=base['path'], base_digest=base['digest'],
                checks=dict(service={**base['service_checks'], 'tlt_window_within_deadline': True},
                            compatibility=dict(compatible=True, disjoint_from=comparisons)))
    artifact, artifact_digest = write_artifact(base['output'], info)
    return Candidate(departure, offer_id, dict(status='QUALIFIED', artifact=artifact,
                                               artifact_digest=artifact_digest, info=info))


def admit(request, ledger, base):
    # Frozen snapshot: the standing commitments C_cmt = <T | L> stay fixed for this attempt.
    ledger = tuple(ledger)
    commitment, snapshot = standing(ledger), snapshot_id(ledger)
    tried = []
    for service in request.services:  # the original service, then permitted adaptations
        S_a, B, S_u, C_client, R_task = service_contracts(service)

        # III-B, owner obligations: Q and its access-only guarantee.
        Q = R_task.quotient(C_client).normalize()
        assert Q.assumption == normalize(~S_a | (B & S_u))
        assert Q.guarantee == normalize(S_a)

        # III-B, retain standing obligations: C_plus = C_cmt ∧ Q = <T | L ∧ S_a>.
        C_plus = commitment.conjoin(Q)
        assert normalize(C_plus.guarantee) == normalize(commitment.guarantee & S_a)

        # Address L ∧ S_a within the finite family, template by template.
        candidates = tuple(qualify(service, s, snapshot, ledger, base) for s in service.departures)
        tried.append(ServiceAdmission(service, S_a, B, S_u, C_client, R_task, Q, C_plus, candidates))
        if any(c.status == 'QUALIFIED' for c in candidates):
            break  # freeze ONE service and client model before generating offers
    return Admission(request, snapshot, commitment, tuple(tried))

**Offer template [semantic, then algebra].** For a qualified departure $s$, $H_\kappa$ means
$r_j(t)\supseteq R_\kappa(t)$ throughout the lease. The swept-body check gives
$$\operatorname{Body}(\xi_s(t))\subseteq R_\kappa(t)\ \ \forall t\in[s,s+7]
\quad\Longrightarrow\quad H_\kappa\Rightarrow a_s\Rightarrow S_a .$$
The offer stores the explicit consequence $K_\kappa=\langle\top\mid H_\kappa\land a_s\rangle$.
Its implication to $S_a$ is therefore algebraic, but the *redundancy* of $a_s$ given
$H_\kappa$ is the geometric fact: Boolean normalization cannot infer containment between
unrelated proposition names. `certify_service` checks the containment evidence before the
implication.

**Compatibility [semantic, numerical].** Reservations that are disjoint in full space–time
(closed cells and neighbouring time slabs) admit the allocation $r_i:=R_{\kappa_i}$ for every
standing BLA and $r_j:=R_\kappa$. That allocation satisfies $L\land H_\kappa$.
`certify_compatibility` checks this separately: neither certificate substitutes for the other.

**Refinement and composition [algebra].** $K_\kappa\preceq Q$; conjunction monotonicity and
the quotient adjunction then give
$$C_{\rm cmt}\wedge K_\kappa\preceq C_{\rm cmt}^{+},\qquad
C_{\rm client}\otimes(C_{\rm cmt}\wedge K_\kappa)\preceq C_{\rm cmt}\wedge R_{\rm task}.$$
These are statements about the atoms. Their physical reading rests on the client model:
the fixed policy, the exact entry state and perfect nominal dynamics.

In [15]:
def generate(admission):
    # III-C: one offer per qualified template of the single selected service.
    selected = admission.selected
    offers = []
    for candidate in (() if selected is None else selected.qualified):
        info = candidate.reply['info']
        offer = CorridorOffer(candidate.offer_id, selected.service, selected.S_a, candidate.departure,
                              F(info['end']), admission.snapshot, candidate.reply['artifact'],
                              candidate.reply['artifact_digest'], json.dumps(info['checks'], sort_keys=True))
        certify_service(offer)        # containment evidence first, then K_kappa ⇒ S_a
        certify_compatibility(offer)  # disjoint from every standing reservation
        offers.append(offer)
    return tuple(offers)  # frozen: one service, one client model, one snapshot

**Bargaining.** `bargain` is the function body from `protocol.ipynb`, compiled verbatim. Only
its offer-eligibility validator is replaced by `validate_corridors`, which requires both
certificates, one service, one client model and one snapshot. Utilities $u_o=q-c_\kappa$ and $u_c=v_\kappa-q$
are frozen before the first round, and no HJ solve runs during bargaining.

**Commitment.** Store the exact accepted object and price, then
$C_{\rm cmt}\leftarrow C_{\rm cmt}\wedge K^*$. Stale snapshots, substituted objects and
changed artifacts are rejected. Without an agreement the commitments are unchanged and the
outcome keeps its reason: price failure, rejected templates, unresolved evidence or proved
impossibility. Unused allocations remain promised until lease end.

In [16]:
def commit(admission, offers, negotiation, ledger):
    ledger = tuple(ledger)
    if negotiation.agreement is None:
        # III-D: no agreement keeps every commitment; the outcome keeps its reason.
        status = 'PRICE-NO-BLA' if offers else admission.failure
        return CommitResult(status, admission.commitment, ledger, None)
    # Exact accepted object, fresh snapshot, both certificates and unchanged artifacts.
    updated = commit_exact(offers, negotiation, ledger, verify_artifact)
    return CommitResult('COMMITTED', admission.commitment.conjoin(updated[-1].contract),
                        updated, updated[-1])


def attempt(spec, ledger, base):
    # admit → generate → bargain → commit, unchanged, for one request.
    request, previous = spec['request'], tuple(ledger)
    for bla in previous:
        verify_artifact(bla.offer)
    admission = admit(request, previous, base)
    offers = generate(admission)
    policies = affine_policies(dict(zip(offers, spec['costs'])), dict(zip(offers, spec['values'])))
    negotiation = bargain(offers, policies, spec['targets'], spec['minima'], PRICES, ROUND_BUDGET)
    result = commit(admission, offers, negotiation, previous)
    # Every earlier BLA object and artifact is retained, whatever the outcome.
    preserved = len(result.ledger) >= len(previous) and all(a is b for a, b in zip(result.ledger, previous))
    for bla in previous:
        verify_artifact(bla.offer)
    assert preserved and (result.status == 'COMMITTED' or result.ledger is previous)
    return dict(request=request, admission=admission, offers=offers, policies=policies,
                negotiation=negotiation, result=result, previous_preserved=preserved)

## 4. Staged execution

Five serial requests, all negotiated at frozen virtual decision time zero, before execution.

| Client | Intended case |
|:--|:--|
| A | First accepted eastbound crossing; the initial standing commitment. |
| B | Two westbound alternatives for one fixed service, overlapping A's lease in time. |
| C | Same-lane templates that conflict with A's retained reservation. |
| D | Compatible later templates, rejected through price bargaining. |
| E | Original deadline excluded by an independent speed bound; a permitted later service is admitted. |

Early/later offers carry illustrative costs 5/2 and values 10/8; D has values 1/0. These are
declared preferences that create a price/delay choice, not fitted traffic economics.

In [17]:
PRICES, ROUND_BUDGET = (F(0), F(20)), 3
PREFERENCES = dict(costs=(F(5), F(2)), values=(F(10), F(8)),
                   targets={'owner': F(4), 'client': F(4)}, minima={'owner': F(1), 'client': F(1)})
PRICE_FAILURE = dict(costs=(F(5), F(2)), values=(F(1), F(0)),
                     targets={'owner': F(4), 'client': F(1)}, minima={'owner': F(2), 'client': F(0)})
REQUESTS = (
    dict(request=CrossingRequest('A', CrossingService('A', 'east', (0,), 8)), **PREFERENCES),
    dict(request=CrossingRequest('B', CrossingService('B', 'west', (0, 1), 9)), **PREFERENCES),
    dict(request=CrossingRequest('C', CrossingService('C', 'east', (0, 1), 9)), **PREFERENCES),
    dict(request=CrossingRequest('D', CrossingService('D', 'west', (10, 11), 19)), **PRICE_FAILURE),
    dict(request=CrossingRequest('E', CrossingService('E', 'east', (20,), 22),
                                 alternatives=(CrossingService('E', 'east', (20,), 28),)), **PREFERENCES),
)

In [18]:
table(['Service', 'Departure s', 'Deadline d', 'In S_a: s+7 ≤ d', 'TLT window applies: s+8 ≤ d',
       'Speed bound excludes [independent]'],
      [(f'{svc.client} {svc.route}', s, svc.deadline, s in eligible(svc),
        tlt_evidence_applies(s, svc.deadline), speed_bound_excludes(s, svc.deadline))
       for spec in REQUESTS for svc in spec['request'].services for s in svc.departures])

| Service | Departure s | Deadline d | In S_a: s+7 ≤ d | TLT window applies: s+8 ≤ d | Speed bound excludes [independent] |
| --- | --- | --- | --- | --- | --- |
| A east | 0 | 8 | True | True | False |
| B west | 0 | 9 | True | True | False |
| B west | 1 | 9 | True | True | False |
| C east | 0 | 9 | True | True | False |
| C east | 1 | 9 | True | True | False |
| D west | 10 | 19 | True | True | False |
| D west | 11 | 19 | True | True | False |
| E east | 20 | 22 | False | False | True |
| E east | 20 | 28 | True | True | False |

### A: the initial commitment

A runs through the same stages, aggregated by `attempt`. Its BLA is the standing commitment
that B's attempt freezes.

In [19]:
ledger, attempts = (), []
A = attempt(REQUESTS[0], ledger, BASE)
attempts.append(A)
ledger = A['result'].ledger
assert A['result'].status == 'COMMITTED' and len(ledger) == 1
names = {b.offer.id: b.offer.service.client for b in ledger}
table(['Client', 'Outcome', 'Accepted contract $K^*$', 'Price', 'Lease [s]', '$C_{\\rm cmt}$ after A'],
      [('A', A['result'].status, ledger[-1].contract, ledger[-1].price,
        f'[{ledger[-1].offer.departure}, {ledger[-1].offer.end}]', A['result'].commitment)],
      ledger_symbols(ledger))

| Client | Outcome | Accepted contract $K^*$ | Price | Lease [s] | $C_{\rm cmt}$ after A |
| --- | --- | --- | --- | --- | --- |
| A | COMMITTED | $\displaystyle \left\langle \top \;\middle\vert\; H^{A}_{0} \land a^{A}_{0} \right\rangle$ | 15/2 | [0, 8] | $\displaystyle \left\langle \top \;\middle\vert\; H^{A}_{0} \land a^{A}_{0} \right\rangle$ |

### B, stage 1: contracts and admission against the frozen snapshot

The reader can stop after each of the following cells: every object passed on to the next
stage is displayed first.

In [20]:
B_spec = REQUESTS[1]
frozen = ledger  # A's BLA; fixed for B's whole attempt
admission = admit(B_spec['request'], frozen, BASE)
selected = admission.selected
symbols = ledger_symbols(frozen) | selected.service.symbols
table(['Object [algebra]', 'Value'], [
    ('$S_a$', selected.S_a), ('$B_j$', selected.B), ('$S_u$', selected.S_u),
    (r'$C_{\rm client}$', selected.client), (r'$R_{\rm task}$', selected.task), ('$Q$', selected.quotient),
    (r'$C_{\rm cmt}$ (A)', admission.commitment), (r'$C_{\rm cmt}^{+}$', selected.obligations)], symbols)
assert selected.quotient.assumption == normalize(~selected.S_a | (selected.B & selected.S_u))
assert selected.quotient.guarantee == normalize(selected.S_a)
assert normalize(selected.obligations.guarantee) == normalize(admission.commitment.guarantee & selected.S_a)
assert admission.snapshot == snapshot_id(frozen) and admission.commitment == standing(frozen)

| Object [algebra] | Value |
| --- | --- |
| $S_a$ | $a^{B}_{0} \lor a^{B}_{1}$ |
| $B_j$ | $B_{B}$ |
| $S_u$ | $S_u^{B}(9)$ |
| $C_{\rm client}$ | $\displaystyle \left\langle a^{B}_{0} \lor a^{B}_{1} \;\middle\vert\; B_{B} \land S_u^{B}(9) \lor \neg a^{B}_{0} \land \neg a^{B}_{1} \right\rangle$ |
| $R_{\rm task}$ | $\displaystyle \left\langle \top \;\middle\vert\; B_{B} \land S_u^{B}(9) \right\rangle$ |
| $Q$ | $\displaystyle \left\langle \left(B_{B} \lor \neg a^{B}_{0}\right) \land \left(B_{B} \lor \neg a^{B}_{1}\right) \land \left(S_u^{B}(9) \lor \neg a^{B}_{0}\right) \land \left(S_u^{B}(9) \lor \neg a^{B}_{1}\right) \;\middle\vert\; a^{B}_{0} \lor a^{B}_{1} \right\rangle$ |
| $C_{\rm cmt}$ (A) | $\displaystyle \left\langle \top \;\middle\vert\; H^{A}_{0} \land a^{A}_{0} \right\rangle$ |
| $C_{\rm cmt}^{+}$ | $\displaystyle \left\langle \top \;\middle\vert\; H^{A}_{0} \land a^{A}_{0} \land \left(a^{B}_{0} \lor a^{B}_{1}\right) \right\rangle$ |

In [21]:
table(['Service', 'Departure', 'Status', 'Meaning', 'Service evidence', 'Compatibility', 'Reason'],
      list(candidate_rows(admission, names)))
assert [c.status for c in selected.candidates] == ['QUALIFIED', 'QUALIFIED']

| Service | Departure | Status | Meaning | Service evidence | Compatibility | Reason |
| --- | --- | --- | --- | --- | --- | --- |
| B · d=9 | 0 | QUALIFIED | service and compatibility evidence both hold | True | A: disjoint |  |
| B · d=9 | 1 | QUALIFIED | service and compatibility evidence both hold | True | A: disjoint |  |

### B, stage 2: offers, service evidence and compatibility evidence

Containment evidence is displayed before the algebraic implication to the fixed $S_a$; the
two offers share one service, one client model and one snapshot.

In [22]:
offers = generate(admission)
assert len(offers) == 2 and all(o.service is selected.service and o.access == selected.S_a
                                and o.snapshot == admission.snapshot for o in offers)
symbols |= ledger_symbols((), *offers)
label = lambda o: f'κ(s={o.departure})'
table(['Offer', 'Lease [s]', 'Swept ref. body ⊆ $R_\\kappa$', 'Ref. in tube', 'Entry inside', 'Road', 'Clear',
       'TLT window ⊆ [s, d]'],
      [(label(o), f'[{o.departure}, {o.end}]', *(o.evidence['service'][k] for k in (
          'reference_covered', 'rollout_inside', 'entry_inside', 'road_ok', 'clear_ok', 'tlt_window_within_deadline')))
       for o in offers])
table(['Offer', '$K_\\kappa$', '$K_\\kappa\\Rightarrow S_a$ [algebra]'],
      [(label(o), o.contract, implies(o.contract.guarantee, selected.S_a)) for o in offers], symbols)

| Offer | Lease [s] | Swept ref. body ⊆ $R_\kappa$ | Ref. in tube | Entry inside | Road | Clear | TLT window ⊆ [s, d] |
| --- | --- | --- | --- | --- | --- | --- | --- |
| κ(s=0) | [0, 8] | True | True | True | True | True | True |
| κ(s=1) | [1, 9] | True | True | True | True | True | True |

| Offer | $K_\kappa$ | $K_\kappa\Rightarrow S_a$ [algebra] |
| --- | --- | --- |
| κ(s=0) | $\displaystyle \left\langle \top \;\middle\vert\; H^{B}_{0} \land a^{B}_{0} \right\rangle$ | True |
| κ(s=1) | $\displaystyle \left\langle \top \;\middle\vert\; H^{B}_{1} \land a^{B}_{1} \right\rangle$ | True |

In [23]:
def composition_checks(offer, admission, selected):
    C_cmt, K = admission.commitment, offer.contract
    return dict(K_refines_Q=refines(K, selected.quotient),
                conjunction=refines(C_cmt.conjoin(K), selected.obligations),
                composition=refines(selected.client.compose(C_cmt.conjoin(K)), C_cmt.conjoin(selected.task)))

table(['Offer', 'Disjoint from [numerical]', 'Allocation witness for $L\\land H_\\kappa$ [semantic]',
       '$K_\\kappa\\preceq Q$', '$C_{\\rm cmt}\\wedge K_\\kappa\\preceq C_{\\rm cmt}^{+}$',
       '$C_{\\rm client}\\otimes(C_{\\rm cmt}\\wedge K_\\kappa)\\preceq C_{\\rm cmt}\\wedge R_{\\rm task}$'],
      [(label(o), ', '.join(f"{names[c['offer']]}: {c['disjoint']}" for c in o.evidence['compatibility']['disjoint_from']),
        'r_A := R_A, r_B := R_κ', *composition_checks(o, admission, selected).values()) for o in offers])
assert all(all(composition_checks(o, admission, selected).values()) for o in offers)

| Offer | Disjoint from [numerical] | Allocation witness for $L\land H_\kappa$ [semantic] | $K_\kappa\preceq Q$ | $C_{\rm cmt}\wedge K_\kappa\preceq C_{\rm cmt}^{+}$ | $C_{\rm client}\otimes(C_{\rm cmt}\wedge K_\kappa)\preceq C_{\rm cmt}\wedge R_{\rm task}$ |
| --- | --- | --- | --- | --- | --- |
| κ(s=0) | A: True | r_A := R_A, r_B := R_κ | True | True | True |
| κ(s=1) | A: True | r_A := R_A, r_B := R_κ | True | True | True |

### B, stage 3: bargaining on the frozen family

The invocation, frozen utilities, rounds and accepted pair. The function is the one in
`protocol.ipynb`.

In [24]:
policies = affine_policies(dict(zip(offers, B_spec['costs'])), dict(zip(offers, B_spec['values'])))
table(['Offer', 'Cost $c_\\kappa$', 'Value $v_\\kappa$', 'Targets (owner, client)', 'Minima (owner, client)'],
      [(label(o), B_spec['costs'][i], B_spec['values'][i], tuple(B_spec['targets'].values()),
        tuple(B_spec['minima'].values())) for i, o in enumerate(offers)])
negotiation = bargain(offers, policies, B_spec['targets'], B_spec['minima'], PRICES, ROUND_BUDGET)
table(['Round', 'Proposer', 'Targets (owner, client)', 'Offer', 'Price', 'Utilities (owner, client)', 'Accepted'],
      [(r.number, r.proposer, tuple(v for _, v in r.targets), label(r.proposal.offer), r.proposal.price,
        tuple(v for _, v in r.utilities), r.accepted) for r in negotiation.history])
assert negotiation.status == 'ACCEPTED' and negotiation.offers == offers

| Offer | Cost $c_\kappa$ | Value $v_\kappa$ | Targets (owner, client) | Minima (owner, client) |
| --- | --- | --- | --- | --- |
| κ(s=0) | 5 | 10 | (4, 4) | (1, 1) |
| κ(s=1) | 2 | 8 | (4, 4) | (1, 1) |

| Round | Proposer | Targets (owner, client) | Offer | Price | Utilities (owner, client) | Accepted |
| --- | --- | --- | --- | --- | --- | --- |
| 1 | owner | (4, 4) | κ(s=0) | 9 | (4, 1) | False |
| 2 | client | (5/2, 5/2) | κ(s=0) | 15/2 | (5/2, 5/2) | True |

### B, stage 4: exact commitment

In [25]:
result = commit(admission, offers, negotiation, frozen)
service = selected.service
promised = {s for s in service.departures if implies(result.commitment.guarantee, service.access_for(s))}
departure = select_departure(service, promised)
try:
    commit(admission, offers, negotiation, result.ledger)
    stale = 'accepted'
except ValueError as error:
    stale = f'rejected: {error}'
table(['Commitment item', 'Value'], [
    ('Status', result.status), ('Accepted offer', label(result.agreement.offer)), ('Price', result.agreement.price),
    ('Exact object stored', result.agreement.offer is negotiation.agreement.offer),
    ("A's BLA object retained", result.ledger[0] is frozen[0]),
    ('$C_{\\rm cmt}$ after B', result.commitment),
    ('Departure chosen by the fixed client rule', departure),
    ('Re-committing against the updated snapshot', stale)], ledger_symbols(result.ledger))
assert result.status == 'COMMITTED' and departure == result.agreement.offer.departure
assert stale.startswith('rejected')
B = dict(request=B_spec['request'], admission=admission, offers=offers, policies=policies,
         negotiation=negotiation, result=result,
         previous_preserved=all(a is b for a, b in zip(result.ledger, frozen)))
attempts.append(B)
ledger = result.ledger
names.update({b.offer.id: b.offer.service.client for b in ledger})

| Commitment item | Value |
| --- | --- |
| Status | COMMITTED |
| Accepted offer | κ(s=0) |
| Price | 15/2 |
| Exact object stored | True |
| A's BLA object retained | True |
| $C_{\rm cmt}$ after B | $\displaystyle \left\langle \top \;\middle\vert\; H^{A}_{0} \land H^{B}_{0} \land a^{A}_{0} \land a^{B}_{0} \right\rangle$ |
| Departure chosen by the fixed client rule | 0 |
| Re-committing against the updated snapshot | rejected: stale snapshot or substituted accepted pair |

### C–E with the same stages

`attempt` aggregates the unchanged stages. The candidate table keeps each template's meaning;
E shows its original service and the permitted replacement, each with rebuilt contracts.

In [26]:
for spec in REQUESTS[2:]:
    record = attempt(spec, ledger, BASE)
    attempts.append(record)
    ledger = record['result'].ledger
    names.update({b.offer.id: b.offer.service.client for b in ledger})
C, D, E = attempts[2:]
table(['Service', 'Departure', 'Status', 'Meaning', 'Service evidence', 'Compatibility', 'Reason'],
      [row for a in (C, D, E) for row in candidate_rows(a['admission'], names)])
table(['E service', '$S_a$', '$Q$', 'Templates'],
      [(f'd={s.service.deadline}', s.S_a, s.quotient, ', '.join(c.status for c in s.candidates))
       for s in E['admission'].services], E['admission'].services[-1].service.symbols)
table(['Client', 'Round', 'Proposer', 'Offer', 'Price', 'Utilities (owner, client)', 'Accepted'],
      [(a['request'].client, r.number, r.proposer, label(r.proposal.offer), r.proposal.price,
        tuple(v for _, v in r.utilities), r.accepted) for a in (C, D, E) for r in a['negotiation'].history])
table(['Client', 'Outcome', 'Meaning', 'Earlier BLAs retained', 'BLAs after'],
      [(a['request'].client, a['result'].status, MEANINGS[a['result'].status], a['previous_preserved'],
        len(a['result'].ledger)) for a in attempts])

| Service | Departure | Status | Meaning | Service evidence | Compatibility | Reason |
| --- | --- | --- | --- | --- | --- | --- |
| C · d=9 | 0 | GEOMETRIC_CONFLICT | template rejected: its reservation meets a standing one | True | A: meets, B: disjoint | reservation meets a standing one |
| C · d=9 | 1 | GEOMETRIC_CONFLICT | template rejected: its reservation meets a standing one | True | A: meets, B: disjoint | reservation meets a standing one |
| D · d=19 | 10 | QUALIFIED | service and compatibility evidence both hold | True | A: disjoint, B: disjoint |  |
| D · d=19 | 11 | QUALIFIED | service and compatibility evidence both hold | True | A: disjoint, B: disjoint |  |
| E · d=22 | 20 | PROVED_IMPOSSIBLE | independent necessary bound fails | — | — | 3 m to the exit exceeds vmax · 2 s |
| E · d=28 | 20 | QUALIFIED | service and compatibility evidence both hold | True | A: disjoint, B: disjoint |  |

| E service | $S_a$ | $Q$ | Templates |
| --- | --- | --- | --- |
| d=22 | $\bot$ | $\displaystyle \left\langle \top \;\middle\vert\; \bot \right\rangle$ | PROVED_IMPOSSIBLE |
| d=28 | $a^{E}_{20}$ | $\displaystyle \left\langle \left(B_{E} \lor \neg a^{E}_{20}\right) \land \left(S_u^{E}(28) \lor \neg a^{E}_{20}\right) \;\middle\vert\; a^{E}_{20} \right\rangle$ | QUALIFIED |

| Client | Round | Proposer | Offer | Price | Utilities (owner, client) | Accepted |
| --- | --- | --- | --- | --- | --- | --- |
| D | 1 | owner | κ(s=10) | 9 | (4, -8) | False |
| D | 2 | client | κ(s=10) | 1/2 | (-9/2, 1/2) | False |
| D | 3 | owner | κ(s=10) | 15/2 | (5/2, -13/2) | False |
| E | 1 | owner | κ(s=20) | 9 | (4, 1) | False |
| E | 2 | client | κ(s=20) | 15/2 | (5/2, 5/2) | True |

| Client | Outcome | Meaning | Earlier BLAs retained | BLAs after |
| --- | --- | --- | --- | --- |
| A | COMMITTED | exact negotiated offer and price stored; standing commitment conjoined | True | 1 |
| B | COMMITTED | exact negotiated offer and price stored; standing commitment conjoined | True | 2 |
| C | NO-COMPATIBLE-OFFER | every template conflicts with a retained reservation; not physical impossibility | True | 2 |
| D | PRICE-NO-BLA | qualified offers exist; bargaining exhausted without agreement | True | 2 |
| E | COMMITTED | exact negotiated offer and price stored; standing commitment conjoined | True | 3 |

## 5. Evidence across the numerical/contract boundary

**Accepted behaviour.** For each accepted BLA, the predicates are evaluated under the
allocation witness $r_j:=R_\kappa$ and the client following its selected reference. Their
values come from evidence of the stated strength, not from Boolean normalization. References
that no agreement promises are evaluated as uncovered (conservatively). The owner promise
$L$ is the whole standing guarantee.

In [27]:
def evidence_values(ledger):
    # Allocation witness: all accepted reservations pairwise disjoint (each offer was checked
    # against exactly the BLAs before it), so r_j := R_kappa_j is one allocation.
    values, witness = {}, True
    for i, bla in enumerate(ledger):
        o, e = bla.offer, bla.offer.evidence
        witness &= {c['offer'] for c in e['compatibility']['disjoint_from']} == {b.offer.id for b in ledger[:i]}
        argument = client_argument(o.departure, o.service.deadline)
        a_s = o.service.access_for(o.departure).name
        values[o.promise.name] = witness
        values[a_s] = e['service']['reference_covered']
        values[o.service.rule.name] = values[a_s]  # the client follows the covered ξ_s
        values[o.service.task_atom.name] = (all(argument[k] for k in ('body_in_lane', 'clears', 'timely'))
                                            and e['service']['road_ok'])
    return values


def holds(formula, values):
    assigned = il.as_formula(formula).replace(
        lambda f: il.Boolean(values.get(f.name, False)) if isinstance(f, il.Proposition) else f)
    return normalize(assigned) == il.Boolean(True)


values, L = evidence_values(ledger), standing(ledger).guarantee
behaviour = {}
for bla in ledger:
    o = bla.offer
    S_a, B_j, S_u, C_client, R_task = service_contracts(o.service)
    behaviour[o.service.client] = dict(
        owner=holds(L, values), access=holds(o.service.access_for(o.departure), values),
        assumption=holds(S_a, values), guarantee=holds(C_client.guarantee, values),
        task=holds(R_task.guarantee, values))
evidence = {'owner': '[semantic + numerical] disjoint reservations; r := R is one allocation',
            'access': '[numerical] swept reference-body boxes ⊆ R_κ; H_κ ⇒ a_s',
            'assumption': '[algebra] a_s ⇒ S_a (normalization)',
            'guarantee': '[semantic] fixed policy: covered ξ_s gives B_j; straight motion gives S_u',
            'task': '[semantic + independent] B_j ∧ S_u; clearance by s+7 ≤ d'}
rows = {'owner': 'Owner promise $L$', 'access': 'Reference access $a_s$', 'assumption': 'Client assumption $S_a$',
        'guarantee': 'Client guarantee $(B_j\\land S_u)\\lor\\neg S_a$', 'task': 'Task $B_j\\land S_u$'}
table(['Accepted behaviour', *behaviour, 'Evidence'],
      [(rows[k], *(b[k] for b in behaviour.values()), evidence[k]) for k in rows])
assert all(all(b.values()) for b in behaviour.values())

| Accepted behaviour | A | B | E | Evidence |
| --- | --- | --- | --- | --- |
| Owner promise $L$ | True | True | True | [semantic + numerical] disjoint reservations; r := R is one allocation |
| Reference access $a_s$ | True | True | True | [numerical] swept reference-body boxes ⊆ R_κ; H_κ ⇒ a_s |
| Client assumption $S_a$ | True | True | True | [algebra] a_s ⇒ S_a (normalization) |
| Client guarantee $(B_j\land S_u)\lor\neg S_a$ | True | True | True | [semantic] fixed policy: covered ξ_s gives B_j; straight motion gives S_u |
| Task $B_j\land S_u$ | True | True | True | [semantic + independent] B_j ∧ S_u; clearance by s+7 ≤ d |

**Independent checks.** Simultaneous nominal bodies are separated at all times (the squared
distance between two straight constant-speed paths is quadratic, so its clipped minimizer is
exact). The TLT entry membership is recomputed at half the time step: a sensitivity check, not
an error bound.

In [28]:
def physical_path(offer):
    states = path.copy()
    if offer.service.route == 'west':  # rotation by π
        states[:, :2] *= -1
        states[:, 2] += np.pi
    return taus + offer.departure, states

pair_checks = []
for a, b in combinations(ledger, 2):
    gap = minimum_body_gap(*physical_path(a.offer), *physical_path(b.offer), BODY_RADIUS)
    pair_checks.append(dict(first=a.offer.id, second=b.offer.id, minimum_body_gap=None if np.isinf(gap) else gap))
gaps = [p['minimum_body_gap'] for p in pair_checks if p['minimum_body_gap'] is not None]

fine_spec = GridSpec((Axis('t', 0, 9, 73), *grid_spec.axes[1:]), accuracy=grid_spec.accuracy)
fine = LTMSPlanner.bicycle5d(fine_spec, wheelbase=WHEELBASE, **CONTROLS)
fine_tree, fine_inputs = timed_until(fine, road_e, exit_e, TimeWindow(*CLEAR_WINDOW))
fine_tlt, fine_record = cached(OUTPUT, 'tlt-fine', dict(backend(fine_spec), **fine_inputs),
                               lambda: dict(entry=np.asarray(fine_tree.realize(fine.impl))[(0, *entry)]))
assert gaps and min(gaps) > 0 and fine_tlt['entry'] <= TOLERANCE
table(['Check or convention', 'Value'], [
    ('HJ membership', f'value ≤ {TOLERANCE} on the declared grid; time step {DT} s'),
    ('Temporal refinement [numerical]',
     f"time step {DT/2} s: entry value {root[(0, *entry)]:.5f} → {float(fine_tlt['entry']):.5f}"),
    ('Reference rollout', 'sampled every 0.025 s; nearest grid point'),
    ('Reservation contact', 'closed raster cells; neighbouring time slabs also compared'),
    ('Simultaneous bodies [independent]', f'minimum pairwise clearance over continuous time: {min(gaps):.2f} m'),
    ('Leases', 'allocations stay promised on [s, s+8], after the reference exits at s+7')])

| Check or convention | Value |
| --- | --- |
| HJ membership | value ≤ 2e-05 on the declared grid; time step 0.25 s |
| Temporal refinement [numerical] | time step 0.125 s: entry value -0.11535 → -0.11458 |
| Reference rollout | sampled every 0.025 s; nearest grid point |
| Reservation contact | closed raster cells; neighbouring time slabs also compared |
| Simultaneous bodies [independent] | minimum pairwise clearance over continuous time: 0.44 m |
| Leases | allocations stay promised on [s, s+8], after the reference exits at s+7 |

**Failure meanings.** Each case below has a different meaning and consequence. An empty offer
family is not physical impossibility, and an unresolved template is neither an offer nor a
proof.

In [29]:
# Geometric conflict: C's template is certified for service, Boolean-satisfiable with L, yet meets A.
conflict = C['admission'].services[0].candidates[0]
H_C = il.Proposition(f'H_{conflict.offer_id}')
a_C = C['admission'].services[0].service.access_for(conflict.departure)
boolean_sat = normalize(C['admission'].commitment.guarantee & H_C & a_C) != il.Boolean(False)

# Missing coverage: the inflated reservation is generous, so report the delays it does and does
# not cover; an offer whose evidence lacks coverage is rejected before any implication is used.
shifted = {delay: covers_reference(delay) for delay in (1., 2.)}
first = ledger[0].offer
lacking = replace(first, evidence_json=json.dumps(
    {**first.evidence, 'service': {**first.evidence['service'], 'reference_covered': False}}))
try:
    certify_service(lacking)
    coverage_rejected = False
except ValueError:
    coverage_rejected = True

# Unresolved: s+7 ≤ d makes the reference eligible, but the TLT window [s+7, s+8] exceeds d.
boundary = admit(CrossingRequest('F', CrossingService('F', 'east', (30,), 37)), ledger, BASE)
impossible = E['admission'].services[0].candidates[0]

table(['Case', 'Evidence', 'Meaning', 'Consequence'], [
    ('C: geometric conflict', f"service evidence {all(conflict.reply['service'].values())}; "
     f"$L\\land H_\\kappa\\land a_s$ Boolean-satisfiable: {boolean_sat}; reservations meet A's",
     MEANINGS['GEOMETRIC_CONFLICT'], 'empty family: NO-COMPATIBLE-OFFER, not impossibility'),
    ('Missing reference coverage', f"reference delayed 1 s / 2 s covered: {shifted[1.]} / {shifted[2.]}; "
     f"offer without coverage evidence rejected: {coverage_rejected}",
     'H_κ would not imply a_s', 'cannot qualify; the Boolean implication is never consulted'),
    ('F (s=30, d=37): unresolved', f"eligible {30 in eligible(boundary.services[0].service)}; "
     f"status {boundary.services[0].candidates[0].status}",
     MEANINGS['UNRESOLVED'], f"admission failure {boundary.failure}; nothing committed"),
    ('E original (d=22)', impossible.reply['reason'], MEANINGS['PROVED_IMPOSSIBLE'],
     'permitted replacement d=28 admitted with rebuilt contracts'),
    ('D: price failure', f"{len(D['offers'])} qualified offers; {len(D['negotiation'].history)} rejected rounds",
     MEANINGS['PRICE-NO-BLA'], 'commitments unchanged')])
assert conflict.status == 'GEOMETRIC_CONFLICT' and boolean_sat and all(conflict.reply['service'].values())
assert not shifted[2.] and coverage_rejected
assert boundary.selected is None and boundary.failure == 'UNRESOLVED'
assert impossible.status == 'PROVED_IMPOSSIBLE' and E['admission'].selected.service.deadline == 28

| Case | Evidence | Meaning | Consequence |
| --- | --- | --- | --- |
| C: geometric conflict | service evidence True; $L\land H_\kappa\land a_s$ Boolean-satisfiable: True; reservations meet A's | template rejected: its reservation meets a standing one | empty family: NO-COMPATIBLE-OFFER, not impossibility |
| Missing reference coverage | reference delayed 1 s / 2 s covered: True / False; offer without coverage evidence rejected: True | H_κ would not imply a_s | cannot qualify; the Boolean implication is never consulted |
| F (s=30, d=37): unresolved | eligible True; status UNRESOLVED | numerical evidence does not apply; neither an offer nor impossibility | admission failure UNRESOLVED; nothing committed |
| E original (d=22) | 3 m to the exit exceeds vmax · 2 s | independent necessary bound fails | permitted replacement d=28 admitted with rebuilt contracts |
| D: price failure | 2 qualified offers; 3 rejected rounds | qualified offers exist; bargaining exhausted without agreement | commitments unchanged |

**Figure.** Shading is the time union of the *actual* spatial reservations, for visualization
only; compatibility uses their full time-dependent volumes. The right-hand plot fixes heading,
steering and speed and is not a projection over unspecified entry states.

In [30]:
final = dict(
    paths=[dict(id=b.offer.id, route=b.offer.service.route,
                times=physical_path(b.offer)[0].tolist(), states=physical_path(b.offer)[1].tolist(),
                reserved_union=route_reservation(BASE, b.offer.service.route).any(axis=0).T.astype(int).tolist())
           for b in ledger],
    pair_checks=pair_checks, x=x.tolist(), y=y.tolist(), rx=rx.tolist(), ry=ry.tolist(),
    slice=root[0, :, :, entry[2], entry[3], entry[4]].T.tolist())
numerics = dict(tlt=tlt_record, passes=passes_record, base=base_record, refinement=fine_record,
                service_evidence=SERVICE_EVIDENCE)
report = build_report(ROOT, attempts, ledger, final, numerics)
display(Markdown(results_table(report)))
figure = plot_scenario(report)
display_timeline(figure)
export_report(report, figure, OUTPUT)
export_figure(final, report['attempts'], OUTPUT)

| Client | Candidate statuses | Outcome | Meaning | Offers | Selected departure | Price | Prior BLAs retained |
|:--|:--|:--|:--|--:|--:|--:|:--|
| A | d=8: s=0 QUALIFIED | COMMITTED | exact negotiated offer and price stored; standing commitment conjoined | 1 | 0 | 15/2 | True |
| B | d=9: s=0 QUALIFIED, s=1 QUALIFIED | COMMITTED | exact negotiated offer and price stored; standing commitment conjoined | 2 | 0 | 15/2 | True |
| C | d=9: s=0 GEOMETRIC_CONFLICT, s=1 GEOMETRIC_CONFLICT | NO-COMPATIBLE-OFFER | every template conflicts with a retained reservation; not physical impossibility | 0 | — | — | True |
| D | d=19: s=10 QUALIFIED, s=11 QUALIFIED | PRICE-NO-BLA | qualified offers exist; bargaining exhausted without agreement | 2 | — | — | True |
| E | d=22: s=20 PROVED_IMPOSSIBLE; d=28: s=20 QUALIFIED | COMMITTED | exact negotiated offer and price stored; standing commitment conjoined | 1 | 20 | 15/2 | True |

LTMS reachability-based BLAs

**Runner agreement and final checks.** `run_scenario` compiles this notebook's definition
cells and runs A–E serially. It must reproduce the staged walkthrough's outcomes, offer
identities, selected departures, prices and retained promises. Earlier artifacts must be
unchanged afterwards.

In [31]:
runner_attempts, runner_ledger = run_scenario(ROOT, BASE)
assert outcome_summary(runner_attempts) == outcome_summary(attempts)
for bla in ledger:
    verify_artifact(bla.offer)
assert [a['result'].status for a in attempts] == [
    'COMMITTED', 'COMMITTED', 'NO-COMPATIBLE-OFFER', 'PRICE-NO-BLA', 'COMMITTED']
assert all(a['previous_preserved'] for a in attempts)
assert all(a['result'].agreement.offer is a['negotiation'].agreement.offer
           for a in attempts if a['result'].agreement)
assert [b.offer.service.client for b in ledger] == ['A', 'B', 'E']
table(['Client', 'Outcome', 'Offers', 'Selected departure', 'Price', 'Retained BLAs'],
      [(name, status, len(ids), '—' if s is None else s, '—' if q is None else q, len(kept))
       for name, status, ids, s, q, kept in outcome_summary(runner_attempts)])

| Client | Outcome | Offers | Selected departure | Price | Retained BLAs |
| --- | --- | --- | --- | --- | --- |
| A | COMMITTED | 1 | 0 | 15/2 | 1 |
| B | COMMITTED | 2 | 0 | 15/2 | 2 |
| C | NO-COMPATIBLE-OFFER | 0 | — | — | 2 |
| D | PRICE-NO-BLA | 2 | — | — | 2 |
| E | COMMITTED | 1 | 20 | 15/2 | 3 |

## Scope and reproduction

A finite straight-crossing example under perfect initial state and dynamics. The conditional
client interpretation has a concrete nominal controller. Numerical HJ membership and the
sampled corridor construction establish neither general closed-loop invariance, uncertain
tracking nor complete coverage of feasible offers. The temporal refinement is a sensitivity
check, not an error bound, and the route-local computation does not reproduce the
full-resolution legacy LTMS grid.

From the repository root (the runner starts a Python 3.13 kernel through `uv`):

```sh
uv run --group notebooks python scripts/run_ltms_notebook.py --save
# An independent cold numerical run into a new directory:
uv run --group notebooks python scripts/run_ltms_notebook.py --output /tmp/ltms-fresh
```

Outputs include `report.json`, exact offer records, hashed numerical arrays, `results.md`,
an interactive HTML figure and standalone SVG/PNG figures. They are reproducible local
artifacts, excluded from Git.